# M3L2 E00 - Del agente manual (M3L1) a LangChain (M3L2)

## Por que este notebook existe

En M3L1 construimos agentes **a mano**.

Escribimos cada parte: las tools, el loop ReAct, el trace, el limite de pasos.
Funciono. Pero era mucho codigo pegamento.

**M3L2 responde la pregunta**: si ya se como funciona un agente por dentro,
por que no usar un framework que haga el trabajo repetitivo por mi?

Eso es exactamente lo que hace **LangChain**.

Este notebook toma **el mismo ejemplo** del E03 de M3L1 (agente clima + calculo)
y lo implementa dos veces:

1. **Como lo haciamos en M3L1** (manual, 50+ lineas de codigo pegamento)
2. **Como se hace con LangChain** (modular, declarativo, con trace incluido)

---

## Necesita OpenAI API key


## El problema que resuelven los AI Agents (Lecture M3L1 - Seccion 3)

La lecture M3L1 lo plantea asi:

> "Un modelo de lenguaje por si solo tiene limites. No puede conocer datos actuales,
> ejecutar codigo real, verificar cifras en vivo, ni corregirse usando observaciones externas."

La solucion es un agente que puede:

```text
interpretar el objetivo
       |
       v
   decidir pasos
       |
       v
   elegir tools
       |
       v
  ejecutar tools
       |
       v
 observar resultados
       |
       v
 corregir el rumbo
       |
       v
  respuesta final
```

Esa arquitectura la construimos a mano en M3L1. En M3L2 vemos como LangChain la provee.


## El ciclo ReAct que ya conocemos de M3L1

En M3L1 E02 y E03 aprendimos el patron **ReAct** (Reasoning + Acting):

```text
Consulta del usuario
       |
       v
  [Thought]  <- El agente razona que necesita
       |
       v
  [Action]   <- Elige una tool y la ejecuta
       |
       v
[Observation] <- Lee el resultado de la tool
       |
       v
  Tiene suficiente info?
   /          \
  NO           SI
   |            |
  [Thought]    [Final Answer]
  [Action]
[Observation]
```

Este patron tiene **partes fijas** que siempre son las mismas:

| Parte del patron | Lo que escribiamos en M3L1 | Lo que da LangChain |
|---|---|---|
| Registrar pensamiento | `print(f'[Thought] {msg}')` | Automatico con `verbose=True` |
| Registrar accion | `print(f'[Action] {tool}()')` | Automatico con `verbose=True` |
| Registrar observacion | `print(f'[Observation] {r}')` | Automatico con `verbose=True` |
| Loop con max_steps | `for step in range(max_steps):` | `max_iterations=5` en AgentExecutor |
| Elegir siguiente accion | `choose_action(state)` manual | LLM decide con tool binding |
| Mantener trace | `trace = []` + `.append()` | Historial interno del AgentExecutor |
| Actualizar state | `state.update(...)` | Contexto del AgentExecutor |


## PARTE 1: El agente manual de M3L1

Este es el codigo que escribimos en M3L1 E03.

**No es un TODO.** Ejecutalo y observa:
- Cuanto codigo pegamento hay para hacer funcionar el loop
- Que cada etiqueta `[Thought]/[Action]/[Observation]` la escribe el programador
- Que el agente no decide solo que tool usar: la logica esta hardcodeada


In [ ]:
# ===================================================================
# AGENTE MANUAL - Estilo M3L1
# El mismo codigo que construimos en M3L1 E03
# ===================================================================

# --- Tools (funciones Python comunes) ---

WEATHER_DB = {"Paris": 21, "London": 15, "Berlin": 18, "Buenos Aires": 25}


def weather_tool_manual(city: str) -> dict:
    """Tool manual: obtiene temperatura de una ciudad."""
    temp = WEATHER_DB.get(city)
    if temp is None:
        return {"success": False, "error": f"No data for {city}"}
    return {"success": True, "city": city, "temperatura_c": temp}


def calculator_tool_manual(operacion: str, a: float, b: float) -> dict:
    """Tool manual: calculadora segura."""
    ops = {
        "multiplicar": lambda x, y: x * y,
        "sumar": lambda x, y: x + y,
        "restar": lambda x, y: x - y,
    }
    if operacion == "dividir":
        if b == 0:
            return {"success": False, "error": "Division by zero"}
        return {"success": True, "resultado": a / b}
    if operacion not in ops:
        return {"success": False, "error": f"Operacion no soportada: {operacion}"}
    return {"success": True, "resultado": ops[operacion](a, b)}


# --- Funciones de logging del trace (codigo pegamento) ---

def thought(msg): print(f"  [Thought] {msg}")
def action(tool_name, *args): print(f"  [Action]  {tool_name}({', '.join(str(a) for a in args)})")
def observation(result): print(f"  [Observation] {result}")
def final_answer(resp): print(f"  [Final Answer] {resp}"); return resp


# --- El agente: logica hardcodeada, NO decide dinamicamente ---

def agente_manual(consulta: str, ciudad: str) -> str:
    """
    Agente ReAct manual.
    La logica de que tool usar esta HARDCODEADA en el codigo.
    No es el modelo quien decide: somos nosotros.
    """
    print(f"\n[Consulta] {consulta}")
    print()

    # Paso 1: obtener clima (hardcodeado: siempre llamamos weather_tool primero)
    thought(f"Necesito la temperatura de {ciudad}.")
    action("weather_tool", ciudad)
    resultado_clima = weather_tool_manual(ciudad)
    observation(resultado_clima)

    if not resultado_clima["success"]:
        return final_answer(f"No pude obtener el clima de {ciudad}. No voy a estimar.")

    temp = resultado_clima["temperatura_c"]

    # Paso 2: calcular 5x (hardcodeado: siempre multiplicamos por 5)
    thought(f"Ahora debo multiplicar {temp} por 5.")
    action("calculator_tool", "multiplicar", temp, 5)
    resultado_calc = calculator_tool_manual("multiplicar", temp, 5)
    observation(resultado_calc)

    resultado = resultado_calc["resultado"]
    return final_answer(f"La temperatura en {ciudad} es {temp}C y cinco veces eso es {resultado}C.")


# Ejecutar el agente manual
print("=" * 55)
print("AGENTE MANUAL (Estilo M3L1)")
print("=" * 55)
agente_manual("Cual es el clima en Paris y cuanto es 5 veces esa temperatura?", "Paris")


## Problemas del enfoque manual (Lecture M3L2 - Seccion 3.4)

La lecture M3L2 identifica los sintomas de deuda tecnica en scripts manuales:

| Sintoma | En el agente manual de arriba |
|---|---|
| Codigo pegamento | `thought()`, `action()`, `observation()` escritos a mano |
| Logica hardcodeada | El agente SIEMPRE llama weather primero, calc despues. No decide. |
| Difícil cambiar el modelo | `openai.chat.completions.create(...)` esta en todos lados |
| No hay loop real | Dos pasos fijos; si hay 3 preguntas, hay que reescribir |
| Trace manual | Cada `print` lo escribimos nosotros |
| max_steps manual | `for step in range(max_steps)` lo escribimos nosotros |

**El problema central**: nuestro "agente" no es realmente un agente.
Es un pipeline con etiquetas `[Thought]`. El modelo no decide nada.

**Un agente real** necesita que el LLM sea quien razone y elija la tool.
Eso requiere un framework que conecte el LLM con las tools de forma estructurada.

Ahi entra LangChain.


## Lo que LangChain hace por nosotros

La lecture M3L2 (Seccion 5) dice:

> "LangChain permite que un pipeline deje de ser una coleccion de scripts
> y pase a ser un sistema modular."

Esto es el mapa completo de lo que ya sabes de M3L1 y su equivalente en LangChain:

```text
M3L1 (manual)                     M3L2 (LangChain)
---------------------------------  ----------------------------------
def weather_tool(city):            @tool
    ...                            def weather_tool(city: str) -> str:
                                       """Docstring = descripcion"""
                                       ...

choose_action(state)               LLM con tool binding
loop manual                        (el modelo decide cual usar)

for step in range(max_steps):      AgentExecutor(
    ...                                max_iterations=5
                                   )

trace = []                         verbose=True
trace.append({...})                (LangChain loguea automaticamente)

[Thought]/[Action]/[Obs] manual    Mensajes internos del executor

state = {}                         Historial de mensajes del agente
state.update(...)                  (LangChain lo mantiene)
```

**La diferencia clave**: en LangChain, el LLM **realmente decide** que tool usar.
El framework le pasa la descripcion de las tools y el modelo elige segun la consulta.


## Arquitectura del agente LangChain

```text
Consulta del usuario
       |
       v
+------------------+
| AgentExecutor    |  <- orquesta todo, mantiene loop y max_iterations
|  +------------+  |
|  |   Agent    |  |  <- el LLM que decide que tool usar
|  | (ChatOpenAI|  |
|  | + prompt)  |  |
|  +-----+------+  |
|        |          |
|  decide tool      |
|        |          |
|  +-----v------+  |
|  |   Tools    |  |  <- funciones decoradas con @tool
|  | [weather]  |  |     LangChain lee el docstring y genera el schema
|  | [calculator]|  |
|  +------------+  |
+------------------+
       |
       v
  Respuesta final
```

Con `verbose=True`, el AgentExecutor imprime automaticamente:
- Que herramienta eligio el modelo
- Con que argumentos la llamo
- Que devolvio la herramienta
- La respuesta final

Todo eso que en M3L1 escribiamos con `thought()`, `action()`, `observation()`.


In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"LLM listo: {llm.model_name}")


## TODO 1: Decorar las tools con `@tool`

El decorator `@tool` hace tres cosas que en M3L1 haciamos a mano:

1. **Lee el docstring** y lo usa como descripcion para el modelo
   (en M3L1 haciamos: `{"description": "Retrieves inventory..."}` manual)

2. **Lee el type hint** y genera el schema de parametros automaticamente
   (en M3L1 haciamos: `{"parameters": {"type": "object", "properties": {...}}}` manual)

3. **Registra la funcion** para que el AgentExecutor pueda invocarla

**Por que el docstring es critico** (Lecture M3L2 - Seccion 16.3):

> "La descripcion no es decorativa. La descripcion ayuda al modelo a decidir cuando usar esa tool."
>
> Mala: `Gets product info.`
> Buena: `Retrieves current inventory level for a product using its SKU.`

Si el docstring es vago, el modelo puede no saber cuando llamar esa tool.


In [ ]:
# TODO 1: Decorar las tools con @tool
# El docstring es la descripcion que el modelo va a leer para decidir cuando usar la tool

# TODO 1a: completar esta tool con @tool y un buen docstring
# @tool
def weather_tool(city: str) -> str:
    # TODO: agregar docstring que explique que hace esta tool
    # y cuando debe usarse
    temps = {"Paris": 21, "London": 15, "Berlin": 18, "Buenos Aires": 25}
    temp = temps.get(city)
    if temp is None:
        return f"No temperature data available for {city}"
    return f"{temp} Celsius"


# TODO 1b: completar esta tool con @tool y un buen docstring
# @tool
def calculator_tool(operation: str, a: float, b: float) -> float:
    # TODO: agregar docstring que explique operaciones disponibles
    ops = {
        "multiply": lambda x, y: x * y,
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
    }
    if operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b
    if operation not in ops:
        raise ValueError(f"Unsupported operation: {operation}. Use: multiply, add, subtract, divide")
    return ops[operation](a, b)


# Verificar que el decorator funciono
# Si el decorator esta bien puesto, la tool tiene .name y .description
# print(f"weather_tool.name: {weather_tool.name}")
# print(f"weather_tool.description: {weather_tool.description}")
# print()
# print(f"calculator_tool.name: {calculator_tool.name}")
# print(f"calculator_tool.description: {calculator_tool.description}")


## TODO 2: Crear el agente con `create_tool_calling_agent`

`create_tool_calling_agent` reemplaza todo lo que en M3L1 era:
- `choose_action(state)`: el LLM decide solo
- El loop `for step in range(max_steps)`: lo hace AgentExecutor
- El trace `trace = []`: lo hace AgentExecutor con `verbose=True`

El prompt necesita un `MessagesPlaceholder(variable_name="agent_scratchpad")`
que es donde LangChain va guardando el historial de tools usadas.
Es el equivalente del `trace = []` que haciamos en M3L1.


In [ ]:
# Prompt del agente
# agent_scratchpad es donde LangChain guarda el historial de acciones
# (equivalente al trace = [] de M3L1)
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools to answer questions accurately."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

tools = [weather_tool, calculator_tool]

# TODO 2a: crear el agente con create_tool_calling_agent(llm, tools, agent_prompt)
# agent = ...
agent = None  # reemplazar

# TODO 2b: crear el ejecutor con AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)
# executor = ...
executor = None  # reemplazar

print(f"Agent: {type(agent).__name__ if agent else 'TODO no completado'}")
print(f"Executor: {type(executor).__name__ if executor else 'TODO no completado'}")
print()
print("max_iterations=5 es el equivalente del max_steps=5 de M3L1")
print("verbose=True reemplaza todos los prints de [Thought]/[Action]/[Observation]")


## TODO 3: Invocar el agente y comparar

El agente LangChain recibe la consulta en lenguaje natural y decide solo:
- Que tools usar
- Con que argumentos
- Cuantas veces
- Cuando terminar

Con `verbose=True` veras el trace automatico, sin haber escrito ningun `print()`.


In [ ]:
# TODO 3: invocar el agente y ver el trace automatico

if executor:
    print("=" * 55)
    print("AGENTE LANGCHAIN (Estilo M3L2)")
    print("=" * 55)
    # TODO: invocar el executor con esta consulta
    # result = executor.invoke({"input": "What is the weather in Paris and what is 5 times that temperature?"})
    # print()
    # print(f"Respuesta final: {result['output']}")
else:
    print("Completar TODO 2 primero")


## Comparacion side-by-side: M3L1 vs M3L2

Ejecuta ambas versiones y compara el output.


In [ ]:
print("=" * 55)
print("COMPARACION: M3L1 manual vs M3L2 LangChain")
print("=" * 55)
print()
print("M3L1 - Lineas de codigo pegamento necesarias:")
pegamento_m3l1 = [
    "def thought(msg): print(f'[Thought] {msg}')",
    "def action(tool, *args): print(f'[Action] ...')",
    "def observation(r): print(f'[Observation] {r}')",
    "def final_answer(r): ...",
    "trace = []",
    "for step in range(max_steps):",
    "    action_name = choose_action(state)",
    "    result = TOOLS[action_name](state)",
    "    trace.append({...})",
    "    state.update(...)",
]
for linea in pegamento_m3l1:
    print(f"  {linea}")
print(f"\nTotal: ~{len(pegamento_m3l1)} bloques de codigo pegamento")
print()
print("M3L2 - LangChain hace todo eso con:")
langchain_equiv = [
    "AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)",
]
for linea in langchain_equiv:
    print(f"  {linea}")
print(f"\nTotal: 1 linea de configuracion")
print()
print("El resultado es el mismo. El agente LangChain ademas:")
print("  - Puede manejar N tools (no solo 2)")
print("  - El LLM decide dinamicamente cual usar")
print("  - Puede hacer N pasos (no solo los que hardcodeamos)")


In [ ]:
def run_checks():
    # Verificar que las tools tienen el decorator
    assert hasattr(weather_tool, 'name'), "weather_tool debe tener @tool decorator"
    assert hasattr(calculator_tool, 'name'), "calculator_tool debe tener @tool decorator"
    assert weather_tool.name == "weather_tool"
    assert calculator_tool.name == "calculator_tool"

    # Verificar que tienen descripcion (el docstring)
    assert weather_tool.description, "weather_tool necesita un docstring como descripcion"
    assert calculator_tool.description, "calculator_tool necesita un docstring como descripcion"

    # Verificar que el agente y executor existen
    assert agent is not None, "TODO 2a: agent es None"
    assert executor is not None, "TODO 2b: executor es None"

    # Verificar que el executor funciona con una pregunta simple
    result = executor.invoke({"input": "What is the weather in Paris?"})
    assert "output" in result, "El resultado debe tener la clave 'output'"
    assert isinstance(result["output"], str) and len(result["output"]) > 0
    assert "21" in result["output"] or "celsius" in result["output"].lower() or "paris" in result["output"].lower()

    print("M3L2 E00 Starter checks passed")


run_checks()


## Cierre: el mapa completo M3L1 → M3L2

| Lo que aprendiste en M3L1 | Su equivalente en LangChain | Notebook M3L2 |
|---|---|---|
| f-string como prompt | `ChatPromptTemplate` | E01 PromptTemplate |
| `openai.chat.completions.create()` | `ChatOpenAI` | E03 LLM Wrapper |
| `response.choices[0].message.content` | `StrOutputParser` | E04 OutputParser |
| Logica del loop manual | `AgentExecutor` | Este notebook |
| `choose_action(state)` | LLM con tool binding | Este notebook |
| `trace = []` + prints | `verbose=True` | Este notebook |
| `def weather_tool(city)` | `@tool def weather_tool(city: str)` | Este notebook |
| `max_steps = 5` | `max_iterations=5` | Este notebook |
| `FAISS.from_texts` + busqueda naive | `FAISS` + `Retriever` | E06, E07 |
| Pipeline RAG completo | `LCEL chain` | E01, E02 RAG |

LangChain no inventa conceptos nuevos.
Toma los conceptos que ya conoces de M3L1 y los abstrae en componentes reutilizables.

La diferencia no es que el agente LangChain sea magico.
La diferencia es que puedes cambiar cualquier componente sin reescribir el resto.
